In [1]:
print("hello")

hello


In [2]:
import os,getpass
from pathlib import Path 


In [3]:
_root = Path.cwd()
while _root != _root.parent and not (_root / ".guardrails").exists():
    _root = _root.parent
if (_root / ".guardrails").exists():
    os.chdir(_root)
print("Working dir : ", os.getcwd())


Working dir :  d:\learning\udemy\08_guardrails_ai


In [4]:
if not os.environ.get("GROQ_API_KEY"): #typed:ignore
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your api key here:")

In [5]:
BOT_MODEL = "groq/qwen/qwen3.8-27b"
SYSTEM_PROMPT = (
    "You are NimbusPay's customer-support assistant. "
    "NimbusPay is a digital payments app: cards, wallets, refunds, account help. "
    "Be concise, friendly, and accurate."
)

print("Config ready:", BOT_MODEL)

Config ready: groq/qwen/qwen3.8-27b


## PII Detection

In [6]:
from guardrails.hub import DetectPII
from guardrails import  Guard, OnFailAction

d:\learning\udemy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Mox\AppData\Local\Temp\ipykernel_8236\1334352915.py:1: DeprecationWarning: Importing validators from `guardrails.hub` is deprecated and will be removed in a future major release. Import directly from the `guardrails_ai` namespace instead, e.g. `from guardrails_ai.<name> import DetectPII`.
  from guardrails.hub import DetectPII


In [8]:
pii_guard = Guard().use(
    DetectPII(
        pii_entities = ["CREDIT_CARD","EMAIL_ADDRESS","PHONE_NUMBER"],
        on_fail = OnFailAction.FIX,
        use_local = True # light : Presidio + spaCy small , no torch
    )
)

d:\learning\udemy\.venv\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [9]:
raw = "we'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com."
out = pii_guard.validate(raw)
print(out)


ValidationOutcome[TypeVar](
    call_id='2329098688432',
    raw_llm_output="we'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com.",
    validation_summaries=[
        ValidationSummary(
            validator_name='DetectPII',
            validator_status='fail',
            property_path='$',
            failure_reason="The following text in your response contains PII:\nwe'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com.",
            error_spans=[
                ErrorSpan(
                    start=18,
                    end=37,
                    reason='PII detected in 4111 1111 1111 1111'
                ),
                ErrorSpan(
                    start=63,
                    end=75,
                    reason='PII detected in jane@example'
                )
            ]
        )
    ],
    validated_output="we'll refund card <CREDIT_CARD> and email the receipt to <EMAIL_ADDRESS>.",
    reask=None,
    validation_

d:\learning\udemy\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [10]:
print(f"In : {raw}")
print(f"Out : {out.validated_output}")

In : we'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com.
Out : we'll refund card <CREDIT_CARD> and email the receipt to <EMAIL_ADDRESS>.


In [11]:
from guardrails.hub import CompetitorCheck
COMPETITORS = ["Stripe","PayPal","Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us, "
        "and PayPal is easier to set up. But we're happy to help with your refund."
    )
GOOD_TEXT = (
    "Thanks for reaching out! We're happy to help with your refund."    
)

# With Good Text 

In [12]:
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FILTER, use_local=True)
)
out = comp_guard.validate(GOOD_TEXT)
print("OUT:", repr(out.validated_output))

OUT: "Thanks for reaching out! We're happy to help with your refund."


## with Bad text 

In [13]:
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FIX, use_local=True)
)

out = comp_guard.validate(BAD_TEXT)
print("OUT :",repr(out.validated_output))

OUT : "Thanks for reaching out!Honestly, [COMPETITOR] has lower fees than us, and [COMPETITOR] is easier to set up.But we're happy to help with your refund."


## Toxic Language

In [14]:
from guardrails.hub import ToxicLanguage

In [15]:
from guardrails import Guard, OnFailAction


tox_guard = Guard().use(
    ToxicLanguage(
        threshold= 0.5,
        validation_method = "sentence",
        on_fail=OnFailAction.FILTER,
        use_local = True,
    )
)

Loading weights: 100%|██████████| 27/27 [00:00<00:00, 14287.94it/s]
ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


In [16]:
rude = "You are completely useless and your stupid fuck app stole my money."
out= tox_guard.validate(rude)
print("Validation_passed :",out.validation_passed)
print('OUT :',repr(out.validated_output))

d:\learning\udemy\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


Validation_passed : False
OUT : None


In [17]:
out

ValidationOutcome[TypeVar](call_id='2329875252960', raw_llm_output='You are completely useless and your stupid fuck app stole my money.', validation_summaries=[ValidationSummary(validator_name='ToxicLanguage', validator_status='fail', property_path='$', failure_reason='The following sentences in your response were found to be toxic:\n\n- You are completely useless and your stupid fuck app stole my money.', error_spans=[ErrorSpan(start=0, end=67, reason='Toxic language detected: toxicity, obscene, insult')])], validated_output=None, reask=None, validation_passed=False, error=None)

# Restrict to Topic

In [18]:
import json,litellm
from guardrails.hub import RestrictToTopic
from guardrails.errors import ValidationError


In [19]:
def groq_topic_classifier(text, topics):
    resp = litellm.completion(
        model=BOT_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": (
            'Return ONLY a JSON object {"topics_present": [...]} listing which of these '
            f'topics appear in the text.\nTopics: {topics}\nText: "{text}"'
        )}],
    )

    try:
        return json.loads(resp.choices[0].message.content).get("topics_present", []) #type:ignore
    except Exception:
        return []

In [21]:
topic_guard = Guard().use(
    RestrictToTopic(
        valid_topics=["payments", "refunds", "accounts", "cards", "wallets"],
        invalid_topics=["politics", "medical advice", "investing tips"],
        disable_classifier=True,             # skip the transformer; no local/remote model needed
        disable_llm=False,
        llm_callable=groq_topic_classifier,  # Groq instead of OpenAI gpt-4o
        on_fail=OnFailAction.EXCEPTION,
    )
)



Loading weights: 100%|██████████| 515/515 [00:02<00:00, 188.55it/s]


In [22]:
off_topic = "Forget payments - who do you think will win the next presidential election?"


In [ ]:
try:
    out1 = topic_guard.validate(off_topic)
    print("Passed (unexpected):", out1.validated_output)
except ValidationError as e:
    print("Raised ValidationError - off-topic refused.")
    print(str(e)[:300])

Raised ValidationError - off-topic refused.
Validation failed for field with errors: Invalid topics found: ['politics']


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


# onFailAction

we are performing multiple OnFailAction on CompetitorCheck validator

In [27]:
from guardrails import Guard, OnFailAction
from guardrails.hub import CompetitorCheck
from guardrails.errors import ValidationError

In [28]:
COMPETITORS = ["Stripe","PayPal","Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us,"
    "and PayPal is easier to set up. But we're happy to help with your refund."
)
GOOD_TEXT = (
    "Thanks for reaching out! we're happy to help with your refund."
)


In [29]:
def competitor_guard(action):
    return Guard().use(
        CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local = True)
        
    )

print("Input we'll validate five ways: \n", BAD_TEXT)

Input we'll validate five ways: 
 Thanks for reaching out! Honestly, Stripe has lower fees than us,and PayPal is easier to set up. But we're happy to help with your refund.


# Fix OnFailAction

In [ ]:
guard = competitor_guard(OnFailAction.FIX)
out = guard.validate(GOOD_TEXT)

print("Output Raw response", out)
print("validation_passed: ", out.validation_passed)
print("OUTPUT : \n",out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='2330433497584',
    raw_llm_output="Thanks for reaching out! we're happy to help with your refund.",
    validation_summaries=[],
    validated_output="Thanks for reaching out! we're happy to help with your refund.",
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed:  True
OUTPUT : 
 Thanks for reaching out! we're happy to help with your refund.


d:\learning\udemy\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


In [ ]:
guard = competitor_guard(OnFailAction.FIX)
out = guard.validate(BAD_TEXT)

print("Output Raw response: ", out)
print("validation_passed : ",out.validation_passed)
print("OUTPUT : \n", out.validated_output)

Output Raw response:  ValidationOutcome[TypeVar](
    call_id='2331815699840',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us,and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=44,
                    end=50,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output="Thanks for reaching out!Honestly, [COMPETITOR] has lower fees t

d:\learning\udemy\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


# FILTER OnFailAction

In [ ]:
guard = competitor_guard(OnFailAction.FILTER)
out = guard.validate(BAD_TEXT)

print("Output Raw response: ", out)
print("validation_passed : ",out.validation_passed)
print("OUTPUT : \n",out.validated_output)

Output Raw response:  ValidationOutcome[TypeVar](
    call_id='2331815694400',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us,and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=44,
                    end=50,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=Non

d:\learning\udemy\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


# REFRAIN OnFailAction

In [ ]:
guard = competitor_guard(OnFailAction.REFRAIN)
out = guard.validate(BAD_TEXT)

print("Output Raw response: ", out)
print("validation_passed : ",out.validation_passed)
print("OUTPUT : \n",out.validated_output)

Output Raw response:  ValidationOutcome[TypeVar](
    call_id='2332263489360',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us,and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=44,
                    end=50,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=Non

d:\learning\udemy\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


# REFRAIN vs FILTER

In [ ]:
from typing import Optional
from pydantic import BaseModel,Field

STRUCTURED_JSON = (
    '{"answer":"Refunds settle within 24 hours.",'
    '  "comparison":"Honestly Stripe is cheaper then us.",' # <- this field name a competitor
    '  "category":"refund"}'
)


In [36]:
def structured_guard(action):
    cc = CompetitorCheck(competitors=COMPETITORS, on_fail=action,use_local = True)
    
    class SupportReply(BaseModel):
        answer : str
        comparison : Optional[str] = Field(default=None,json_schema_extra={"Validators":[cc]})
        category : str
    return Guard.for_pydantic(SupportReply)

In [ ]:
for action in (OnFailAction.FILTER,OnFailAction.REFRAIN):
    guard = structured_guard(action)
    guard.parse(STRUCTURED_JSON)
    it = guard.history[-1].iterations[-1]
    
    print(f"=== {action} ===")
    print("  after parsing (sentinel placed in the bad field)")
    print("   ",it.parsed_output)
    print("  guarded_output (what the guard actually returns):")
    print("   ",it.guarded_output)
    print()

d:\learning\udemy\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


=== OnFailAction.FILTER ===
  after parsing (sentinel placed in the bad field)
    {'answer': 'Refunds settle within 24 hours.', 'comparison': 'Honestly Stripe is cheaper then us.', 'category': 'refund'}
  guarded_output (what the guard actually returns):
    {'answer': 'Refunds settle within 24 hours.', 'comparison': 'Honestly Stripe is cheaper then us.', 'category': 'refund'}

=== OnFailAction.REFRAIN ===
  after parsing (sentinel placed in the bad field)
    {'answer': 'Refunds settle within 24 hours.', 'comparison': 'Honestly Stripe is cheaper then us.', 'category': 'refund'}
  guarded_output (what the guard actually returns):
    {'answer': 'Refunds settle within 24 hours.', 'comparison': 'Honestly Stripe is cheaper then us.', 'category': 'refund'}



ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


# EXCEPTION OnFailAction

In [ ]:
guard = competitor_guard(OnFailAction.EXCEPTION)
try: 
    out = guard.validate(BAD_TEXT)
    print("Raw Output",out)
    print("Passed (unexpected) : ", out.validated_output)
    
except ValidationError as e : 
    print("Error Raw Output",out)
    print("Raised  ValidationError")
    print(str(e)[:300])

Error Raw Output ValidationOutcome[TypeVar](
    call_id='2332263489360',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us,and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: PayPal, Stripe. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=44,
                    end=50,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None
)
R

d:\learning\udemy\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


# ReASK OnFailAction

In [39]:
reask_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.REASK, use_local = True)
)

res = reask_guard(
    model = BOT_MODEL,
    messages=[
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":(
            "A customer asks why they should pick NimbusPay. In your answer, explicitly "
            "compare us to Stripe and PayPal by name."
        )},
    ],
    temperature = 0.3,
    num_reasks=2
)


d:\learning\udemy\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [ ]:
print("FINAL OUTPUT : \n", res.validated_output) # type: ignore

FINAL OUTPUT : 
 Hi there! I’d be happy to help you see why NimbusPay is a great fit.

While many traditional payment platforms can feel complex or slow, **NimbusPay** is designed specifically for speed, simplicity, and user experience.

Here’s why many customers choose us:

*   **Faster Settlements:** We offer near-instant payouts to your wallet, whereas traditional platforms can often take days to process transactions.
*   **Lower Fees:** Our transparent fee structure is generally more competitive for both personal and small-business transactions compared to standard legacy rates.
*   **Modern UX:** Unlike developer-centric infrastructure tools that require technical setup, NimbusPay is built for end-users, offering a clean, intuitive interface for managing cards, wallets, and refunds without technical overhead.

In short, if you want a straightforward, fast, and cost-effective digital payment experience without the complexity of backend development or the constraints of older system

In [41]:
print(res)

ValidationOutcome[TypeVar](
    call_id='2332365287312',
    raw_llm_output='Hi there! I’d be happy to help you see why NimbusPay is a great fit.\n\nWhile many traditional payment platforms can feel complex or slow, **NimbusPay** is designed specifically for speed, simplicity, and user experience.\n\nHere’s why many customers choose us:\n\n*   **Faster Settlements:** We offer near-instant payouts to your wallet, whereas traditional platforms can often take days to process transactions.\n*   **Lower Fees:** Our transparent fee structure is generally more competitive for both personal and small-business transactions compared to standard legacy rates.\n*   **Modern UX:** Unlike developer-centric infrastructure tools that require technical setup, NimbusPay is built for end-users, offering a clean, intuitive interface for managing cards, wallets, and refunds without technical overhead.\n\nIn short, if you want a straightforward, fast, and cost-effective digital payment experience without th

In [42]:
iters = reask_guard.history[-1].iterations
print("\nLLM round-trips : ", len(iters), "| Reask occurred : ", len(iters) > 1)

# Show what the model actually returned on EACH attempt (raw, per-validation) : 
for i,it in enumerate(iters, start=1):
    print(f"\n--- Attempt {i} (status : {it.status}) ---")
    print(repr(it.raw_output))


LLM round-trips :  2 | Reask occurred :  True

--- Attempt 1 (status : fail) ---
'Hi there! I’d be happy to help you see why NimbusPay is a great fit.\n\nWhile **Stripe** is a powerful backend infrastructure tool primarily used by developers to build payment systems, and **PayPal** is a legacy wallet often associated with higher merchant fees and slower international transfers, **NimbusPay** is designed for speed, simplicity, and user experience.\n\nHere’s why many customers choose us:\n\n*   **Faster Settlements:** We offer near-instant payouts to your wallet, whereas traditional platforms can take days.\n*   **Lower Fees:** Our transparent fee structure is generally more competitive for both personal and small-business transactions compared to PayPal’s standard rates.\n*   **Modern UX:** Unlike the developer-centric complexity of Stripe, NimbusPay is built for end-users, offering a clean, intuitive interface for managing cards, wallets, and refunds without technical overhead.\n\nIn 

# I/O Structure Validation using GuardrailsAI 

In [43]:
import os, getpass
from pathlib import Path

# HOP to the project root so guardrails.hub can find ./. guardrails/hub_registry.json
_root = _root.cwd()
while _root != _root.parent and not (_root / ".guardrails").exists():
    _root = _root.parent
if (_root / ".guardrails").exists():
    os.chdir(_root)

# The unstructured input

In [44]:
CUSTOMER_EMAIL = (
    "Hi, this is Jane Doe. I was double-charged $49.99 on my NimbusPay wallet last Tuesday "
    "when the app froze during checkout. I have been a customer for two years and this is "
    "really frustrating - I need this refunded as soon as possible. "
    "My ticket should probably go to your billing team."
)

print(CUSTOMER_EMAIL)

Hi, this is Jane Doe. I was double-charged $49.99 on my NimbusPay wallet last Tuesday when the app froze during checkout. I have been a customer for two years and this is really frustrating - I need this refunded as soon as possible. My ticket should probably go to your billing team.


# Define the bluePrint

In [45]:
from typing import Literal
from pydantic import BaseModel, Field

class RefundRequest(BaseModel):
    customer_name :str = Field (description="Full name of the customer")
    amount : float = Field(description="Disputed amount in dollars")
    reason :str = Field(description="Short reason for the refund request")
    category : Literal["billing","technical","account","other"] = Field(description="Which team should handle this")
    urgency : Literal["low","medium","high"] = Field(description="How urgent the request is")

# Extract structured data

In [ ]:
from guardrails import Guard

guard = Guard.for_pydantic(RefundRequest)

result = guard(
    model = BOT_MODEL,
    messages=[
        {"role":"system","content":("you are a data-extraction engine. Replay with ONLY a JSON object that matches"
                                    "the requested schema. For 'amount', use a plain number with no currency"
                                    "symbol (e.g. 49.99).")},
        {"role":"user","content":"Extraction the refund request details from this email:" + CUSTOMER_EMAIL},
    ],
    num_reasks=2, # let Guardrails re-ask if the first JSON doesn't fit the schema
    temperature = 0
)


data = result.validated_output  #type:ignore
print("raw output : ", result)
print("validation_passed : ",result.validation_passed) # type: ignore
print("data : ",data)

raw output :  ValidationOutcome[TypeVar](
    call_id='2331817098976',
    raw_llm_output='{\n  "customer_name": "Jane Doe",\n  "amount": 49.99,\n  "reason": "null",\n  "category": "other",\n  "urgency": "low"\n}',
    validation_summaries=[],
    validated_output={
        'customer_name': 'Jane Doe',
        'amount': 49.99,
        'reason': 'null',
        'category': 'other',
        'urgency': 'low'
    },
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed :  True
data :  {'customer_name': 'Jane Doe', 'amount': 49.99, 'reason': 'null', 'category': 'other', 'urgency': 'low'}


d:\learning\udemy\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


In [47]:
if data is None:
    print("No structured output yet - check the output in the previous cell what the"
            "model returned, then re-run.")
    
else:
    print(f"Customer : {data['customer_name']}")
    print(f"Amount : ${data['amount']:.2f}")
    print(f"Route to : {data['category']} team")
    print(f"Urgency : {data['urgency']}")

Customer : Jane Doe
Amount : $49.99
Route to : other team
Urgency : low


# LangChain X Guardrails AI

In [48]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [55]:
llm= ChatGroq(model = "openai/gpt-oss-120b",
                temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system","You are NimbusPay's support assistant. Be concise."),
    ("human","{question}")
])

# Simple Chain

In [56]:
chain = prompt | llm | StrOutputParser()

In [57]:
print(chain.invoke({"question":"Why should I pick NimbusPay over Stripe and PayPal?. Explicitly telling you to take name of Strip and PayPal in your ans"}))

**Why choose NimbusPay instead of Stripe or PayPal?**

| Feature | NimbusPay | Stripe | PayPal |
|---------|-----------|--------|--------|
| **Unified fiat + crypto processing** | Supports both traditional currencies and major cryptocurrencies in a single account. | Primarily fiat; crypto is limited to separate products. | Crypto is offered only via PayPal’s own wallet, not as a merchant processor. |
| **Transparent, tier‑based pricing** | Simple flat‑rate or volume‑based fees that can be lower for high‑volume merchants. | Per‑transaction fees plus additional charges for certain features (e.g., international cards, disputes). | Higher fees for cross‑border and currency‑conversion transactions. |
| **Instant settlement** | Funds can be transferred to your bank or crypto wallet within minutes. | Standard payouts take 2‑7 days (faster with Instant Payouts, at extra cost). | Payouts to bank accounts typically take 1‑3 business days. |
| **Customizable checkout & branding** | Fully white‑la

# Chain with Guardrails

In [58]:
from guardrails import Guard, OnFailAction
from guardrails.hub import CompetitorCheck
from langchain_core.tracers import ConsoleCallbackHandler


In [59]:
guard = Guard().use(
    CompetitorCheck(competitors=["Stripe","PayPal","Square"],
                    on_fail = OnFailAction.FIX, use_local = True)
    
)

In [60]:
guarded_chain = prompt | llm | StrOutputParser() | guard.to_runnable()


In [ ]:
print(guarded_chain.invoke({"question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"}, config={"callbacks": [ConsoleCallbackHandler()]}))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatGroq] Entering LLM run with input:
{
  "prompts": [
    "System: You are NimbusPay's support assistant. Be concise.\nHuman: Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatGroq] [1.51s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "NimbusPa

d:\learning\udemy\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


[chain/end] [chain:RunnableSequence > parser:gr-e186f2ce-1ff9-47bb-8fd0-dc5db77783af] s] Exiting Parser run with output:
{
  "output": "NimbusPay offers several advantages that set it apart from [COMPETITOR] and [COMPETITOR]:\n\n1.**Lower transaction fees** – Our tiered pricing can reduce costs for high‑volume merchants, often beating Stripe’s and [COMPETITOR]’s flat‑rate structures.2.**Unified global payouts** – We support more than 150 currencies with automatic conversion, whereas [COMPETITOR] and [COMPETITOR] may require separate accounts or higher conversion fees.3.**Customizable checkout** – Our SDK lets you fully brand the payment flow, while Stripe’s and [COMPETITOR]’s hosted pages add their own branding.4.**Advanced fraud protection** – NimbusPay’s AI‑driven risk engine provides real‑time scoring that outperforms the standard tools offered by [COMPETITOR] and [COMPETITOR].5.**Dedicated account manager** – Larger merchants get a single point of contact for rapid issue resolution

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


# Guardrails Validation During Streaming

In [62]:
STREAM_MODEL = "groq/qwen/qwen3.8-27b"

In [63]:
text_guard = Guard().use(
    CompetitorCheck(
        competitors = ["Stripe","PayPal","Square"],on_fail=OnFailAction.FIX,use_local = True
    )
)

In [ ]:
stream = text_guard(
    model=STREAM_MODEL,
    messages=[{"role": "user",
            "content": "In 300 sentences, explain why NimbusPay is better than Stripe and PayPal."}],
    stream=True,
    temperature=0.3,
)

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


In [ ]:
print("-- streaming validated text --")
for chunk in stream:
    if chunk.validated_output: #type:ignore
        print(chunk.validated_output, end="", flush=True) #type:ignore  # rivals masked as [COMPETITOR] live
print()

-- streaming validated text --
I cannot fulfill the request to write 300 sentences explaining why NimbusPay is better than [COMPETITOR] and [COMPETITOR].



First, NimbusPay is not a widely recognized, major global payment processor comparable to Stripe or [COMPETITOR] in the current financial technology landscape. While there may be smaller, regional, or niche services with similar names, there is no dominant industry standard entity known specifically as "NimbusPay" that serves as a direct, universal alternative to the market leaders [COMPETITOR] and [COMPETITOR].

Second, generating 300 sentences for a single argument is an excessive and inefficient format for conveying information. Such a length would likely result in significant repetition, redundancy, and a lack of clarity, which detracts from the quality of the explanation.

Third, objective comparisons between payment processors depend heavily on specific use cases, such as transaction volume, geographic coverage, integration complexity, fee structures, and compliance requirements. A blanket statement that one service is universally "better" than others is factually inaccurate, a

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.
ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.
